In [1]:
import numpy as np
import math
import random
import csv
import pandas as pd
import scipy
import scipy.optimize as opt
from bayes_opt import BayesianOptimization
from bayes_opt.logger import JSONLogger
from bayes_opt.event import Events

In [2]:
def read_behavioral_data(n):
    result_stay_cue = []
    result_safe_risk = []
    action_stay_cue = []
    action_safe_risk = []
    if_can_ask = []
    fname = './behavioral_data/uncertainty_' + str(n+1) + '_2022.csv'
    with open(fname,'r') as f :
        for line in f.readlines():
            if line.split(',')[0]=='0' or line.split(',')[0]=='1':
                if int(line.split(',')[3])==0:
                    action_stay_cue.append(0)
                elif int(line.split(',')[3])==1 or int(line.split(',')[3])==2:
                    action_stay_cue.append(1)
                else :
                    print('ERROR')
                result_stay_cue.append(int(line.split(',')[3]))
                result_safe_risk.append(int(float(line.split(',')[6])))
                action_safe_risk.append(int(line.split(',')[4]))
                if line.split(',')[1]==' ':
                    if_can_ask.append(0)
                else :
                    if_can_ask.append(1)

    return if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk

In [3]:
def dir(a):
    a_0 = np.sum(a,axis=0)
    A = np.zeros([8,8])
    for i in range(np.shape(A)[0]):
        for j in range(np.shape(A)[1]):
            A[i,j] = a[i,j]/a_0[j]
    return A

def a_update(a,result_stay_cue,result_safe_risk,action_safe_risk,rate):
    if result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0.5,0.5,0,0,0,0,0,0])
        #safeHRC,safeLRC,riskyHRC,riskLRC,stayHRC,stayLRC,cueHRC,cueLRC
        o=np.array([1,0,0,0,0,0,0,0])
        #safe,riskyHR,riskyLR,stay,cueHR,cueLR
    elif result_stay_cue==0 and result_safe_risk==0:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==3:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==9:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])
    elif result_stay_cue==0 and result_safe_risk==12:
        s=np.array([0,0,0.5,0.5,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([1,0,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==0:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==3:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==1 and result_safe_risk==9:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])      
    elif result_stay_cue==1 and result_safe_risk==12:
        s=np.array([0,0,1,0,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==0:
        s=np.array([0,1,0,0,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==0:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,0,1,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==3:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,0,1,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==6 and action_safe_risk==1:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([1,0,0,0,0,0,0,0])
    elif result_stay_cue==2 and result_safe_risk==9:
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,0,1,0,0,0,0,0])    
    else :
        s=np.array([0,0,0,1,0,0,0,0])
        o=np.array([0,1,0,0,0,0,0,0])
    if result_stay_cue!=0:
        a=a+rate*np.outer(o,s)#rate:learning rate
    else :
        a=a+rate*0.1*np.outer(o,s)
    return a

def P_action_stay_cue(A,gamma,action):
    preference = np.array([6,12,9,3,0,0,-1,-1])
    Q_risk_H = np.dot(A[:,2],preference)
    Q_risk_L = np.dot(A[:,3],preference)
    Q_stay = max(6,(Q_risk_H+Q_risk_L)/2)
    Q_cue = (max(5,Q_risk_H-1)+max(5,Q_risk_L-1))/2
    Q_stay_cue = np.array([Q_stay,Q_cue])
    exp_Q = np.sum(np.exp(gamma*Q_stay_cue))
    return (np.exp(gamma*Q_stay_cue)/exp_Q)[action]

def P_action_safe_risk(A,gamma,result_stay_cue,action):
    preference = np.array([6,12,9,3,0,0,-1,-1])
    Q_risk_H = np.dot(A[:,2],preference)
    Q_risk_L = np.dot(A[:,3],preference)
    Q_safe = 6
    if result_stay_cue == 1:
        Q_safe_risk = np.array([Q_safe,Q_risk_H])
    elif result_stay_cue == 2:
        Q_safe_risk = np.array([Q_safe,Q_risk_L])
    else:
        Q_safe_risk = np.array([Q_safe,(Q_risk_H+Q_risk_L)/2])
    exp_Q = np.sum(np.exp(gamma*Q_safe_risk))    
    return (np.exp(gamma*Q_safe_risk)/exp_Q)[action]

def P_model_based_RL(prior,rate,gamma):
    subject = 0
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

In [ ]:
def P_model_based_rl_1(prior,rate,gamma):
    subject = 0
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_1 = BayesianOptimization(
    f=P_model_based_rl_1,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_1 = JSONLogger(path="./logs_m_rl_1.log")
optimizer_m_rl_1.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_1)
optimizer_m_rl_1.maximize(
    init_points=1000,
    n_iter=1000,
)

In [6]:
def P_model_based_rl_2(prior,rate,gamma):
    subject = 1
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_2 = BayesianOptimization(
    f=P_model_based_rl_2,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_2 = JSONLogger(path="./logs_m_rl_2.log")
optimizer_m_rl_2.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_2)
optimizer_m_rl_2.maximize(
    init_points=1000,
    n_iter=1000,
)

5.0


In [ ]:
def P_model_based_rl_3(prior,rate,gamma):
    subject = 2
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_3 = BayesianOptimization(
    f=P_model_based_rl_3,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_3 = JSONLogger(path="./logs_m_rl_3.log")
optimizer_m_rl_3.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_3)
optimizer_m_rl_3.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_4(prior,rate,gamma):
    subject = 3
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_4 = BayesianOptimization(
    f=P_model_based_rl_4,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_4 = JSONLogger(path="./logs_m_rl_4.log")
optimizer_m_rl_4.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_4)
optimizer_m_rl_4.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_5(prior,rate,gamma):
    subject = 4
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_5 = BayesianOptimization(
    f=P_model_based_rl_5,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_5 = JSONLogger(path="./logs_m_rl_5.log")
optimizer_m_rl_5.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_5)
optimizer_m_rl_5.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_6(prior,rate,gamma):
    subject = 5
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_6 = BayesianOptimization(
    f=P_model_based_rl_6,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_6 = JSONLogger(path="./logs_m_rl_6.log")
optimizer_m_rl_6.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_6)
optimizer_m_rl_6.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_6(prior,rate,gamma):
    subject = 5
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_6 = BayesianOptimization(
    f=P_model_based_rl_6,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_6 = JSONLogger(path="./logs_m_rl_6.log")
optimizer_m_rl_6.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_6)
optimizer_m_rl_6.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_7(prior,rate,gamma):
    subject = 6
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_7 = BayesianOptimization(
    f=P_model_based_rl_7,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_7 = JSONLogger(path="./logs_m_rl_7.log")
optimizer_m_rl_7.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_7)
optimizer_m_rl_7.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_8(prior,rate,gamma):
    subject = 7
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_8 = BayesianOptimization(
    f=P_model_based_rl_8,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_8 = JSONLogger(path="./logs_m_rl_8.log")
optimizer_m_rl_8.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_8)
optimizer_m_rl_8.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_9(prior,rate,gamma):
    subject = 8
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_9 = BayesianOptimization(
    f=P_model_based_rl_9,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_9 = JSONLogger(path="./logs_m_rl_9.log")
optimizer_m_rl_9.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_9)
optimizer_m_rl_9.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_9(prior,rate,gamma):
    subject = 8
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_9 = BayesianOptimization(
    f=P_model_based_rl_9,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_9 = JSONLogger(path="./logs_m_rl_9.log")
optimizer_m_rl_9.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_9)
optimizer_m_rl_9.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_10(prior,rate,gamma):
    subject = 9
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_10 = BayesianOptimization(
    f=P_model_based_rl_10,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_10 = JSONLogger(path="./logs_m_rl_10.log")
optimizer_m_rl_10.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_10)
optimizer_m_rl_10.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_11(prior,rate,gamma):
    subject = 10
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_11 = BayesianOptimization(
    f=P_model_based_rl_11,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_11 = JSONLogger(path="./logs_m_rl_11.log")
optimizer_m_rl_11.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_11)
optimizer_m_rl_11.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_12(prior,rate,gamma):
    subject = 11
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_12 = BayesianOptimization(
    f=P_model_based_rl_12,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_12 = JSONLogger(path="./logs_m_rl_12.log")
optimizer_m_rl_12.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_12)
optimizer_m_rl_12.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_13(prior,rate,gamma):
    subject = 12
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_13 = BayesianOptimization(
    f=P_model_based_rl_13,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_13 = JSONLogger(path="./logs_m_rl_13.log")
optimizer_m_rl_13.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_13)
optimizer_m_rl_13.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_14(prior,rate,gamma):
    subject = 13
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_14 = BayesianOptimization(
    f=P_model_based_rl_14,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_14 = JSONLogger(path="./logs_m_rl_14.log")
optimizer_m_rl_14.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_14)
optimizer_m_rl_14.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_15(prior,rate,gamma):
    subject = 14
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_15 = BayesianOptimization(
    f=P_model_based_rl_15,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_15 = JSONLogger(path="./logs_m_rl_15.log")
optimizer_m_rl_15.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_15)
optimizer_m_rl_15.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_16(prior,rate,gamma):
    subject = 15
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_16 = BayesianOptimization(
    f=P_model_based_rl_16,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_16 = JSONLogger(path="./logs_m_rl_16.log")
optimizer_m_rl_16.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_16)
optimizer_m_rl_16.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_17(prior,rate,gamma):
    subject = 16
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_17 = BayesianOptimization(
    f=P_model_based_rl_17,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_17 = JSONLogger(path="./logs_m_rl_17.log")
optimizer_m_rl_17.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_17)
optimizer_m_rl_17.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_18(prior,rate,gamma):
    subject = 17
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_18 = BayesianOptimization(
    f=P_model_based_rl_18,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_18 = JSONLogger(path="./logs_m_rl_18.log")
optimizer_m_rl_18.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_18)
optimizer_m_rl_18.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_19(prior,rate,gamma):
    subject = 18
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_19 = BayesianOptimization(
    f=P_model_based_rl_19,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_19 = JSONLogger(path="./logs_m_rl_19.log")
optimizer_m_rl_19.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_19)
optimizer_m_rl_19.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_20(prior,rate,gamma):
    subject = 19
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_20 = BayesianOptimization(
    f=P_model_based_rl_20,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_20 = JSONLogger(path="./logs_m_rl_20.log")
optimizer_m_rl_20.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_20)
optimizer_m_rl_20.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_21(prior,rate,gamma):
    subject = 20
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_21 = BayesianOptimization(
    f=P_model_based_rl_21,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_21 = JSONLogger(path="./logs_m_rl_21.log")
optimizer_m_rl_21.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_21)
optimizer_m_rl_21.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_22(prior,rate,gamma):
    subject = 21
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_22 = BayesianOptimization(
    f=P_model_based_rl_22,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_22 = JSONLogger(path="./logs_m_rl_22.log")
optimizer_m_rl_22.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_22)
optimizer_m_rl_22.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_23(prior,rate,gamma):
    subject = 22
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_23 = BayesianOptimization(
    f=P_model_based_rl_23,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_23 = JSONLogger(path="./logs_m_rl_23.log")
optimizer_m_rl_23.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_23)
optimizer_m_rl_23.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_24(prior,rate,gamma):
    subject = 23
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_24 = BayesianOptimization(
    f=P_model_based_rl_24,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_24 = JSONLogger(path="./logs_m_rl_24.log")
optimizer_m_rl_24.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_24)
optimizer_m_rl_24.maximize(
    init_points=1000,
    n_iter=1000,
)

In [ ]:
def P_model_based_rl_25(prior,rate,gamma):
    subject = 24
    if_can_ask,action_stay_cue,result_stay_cue,action_safe_risk,result_safe_risk = read_behavioral_data(subject)
    log_p = 0
    a = np.array([[100.0,100,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,prior,prior,0,0,0,0],
                [0,0,0,0,100,100,0,0],
                [0,0,0,0,0,0,100,0],
                [0,0,0,0,0,0,0,100]])
    A = dir(a)
    for i in range(120):
        if if_can_ask[i] == 1:
            log_p += np.log(P_action_stay_cue(A,gamma,action_stay_cue[i]))
        log_p += np.log(P_action_safe_risk(A,gamma,result_stay_cue[i],action_safe_risk[i]))
        a = a_update(a,result_stay_cue[i],result_safe_risk[i],action_safe_risk[i],rate)
        A = dir(a)
    return log_p

pbounds_m_rl = {'prior':(0.1,10),'rate':(0.001,10),'gamma':(0.001,10)}
optimizer_m_rl_25 = BayesianOptimization(
    f=P_model_based_rl_25,
    pbounds=pbounds_m_rl,
    random_state=1,allow_duplicate_points=True)
logger_m_rl_25 = JSONLogger(path="./logs_m_rl_25.log")
optimizer_m_rl_25.subscribe(Events.OPTIMIZATION_STEP, logger_m_rl_25)
optimizer_m_rl_25.maximize(
    init_points=1000,
    n_iter=1000,
)